In [ ]:
from __future__ import annotations

import json
from dataclasses import dataclass, field


@dataclass
class EvaluationResult:

    score: float
    is_correct: bool
    feedback: str
    missing_concepts: list[str] = field(
        default_factory=list
    )
    misconceptions: list[str] = field(
        default_factory=list
    )


class AnswerEvaluator:

    def __init__(self, ai_service):

        self.ai_service = ai_service

    def evaluate(
        self,
        question: str,
        answer: str,
        expected_concepts: list[str],
        evidence_chunks: list,
        user_id: str | None = None,
        project_id: str | None = None,
    ) -> EvaluationResult:

        from app.ai.prompts import (
            EVALUATION_SYSTEM_PROMPT,
        )

        evidence = "\n\n".join(
            chunk.text
            for chunk in evidence_chunks
        )

        prompt = f"""
Evaluate this student's answer.

Question:
{question}

Student answer:
{answer}

Expected concepts:
{json.dumps(expected_concepts)}

Evidence:
{evidence}

Return ONLY JSON:

{{
  "score": 0.0,
  "is_correct": false,
  "feedback": "...",
  "missing_concepts": [],
  "misconceptions": []
}}

Score must be between 0 and 1.
"""

        raw = self.ai_service.generate_text(
            prompt=prompt,
            system_instruction=EVALUATION_SYSTEM_PROMPT,
            operation="answer_evaluation",
            user_id=user_id,
            project_id=project_id,
        )

        data = json.loads(
            self._extract_json(raw)
        )

        score = max(
            0.0,
            min(
                1.0,
                float(data.get("score", 0)),
            ),
        )

        return EvaluationResult(
            score=score,
            is_correct=bool(
                data.get("is_correct", False)
            ),
            feedback=str(
                data.get(
                    "feedback",
                    "",
                )
            ),
            missing_concepts=list(
                data.get(
                    "missing_concepts",
                    [],
                )
            ),
            misconceptions=list(
                data.get(
                    "misconceptions",
                    [],
                )
            ),
        )

    @staticmethod
    def _extract_json(text: str) -> str:

        text = text.strip()

        if text.startswith("```"):
            lines = text.splitlines()

            lines = [
                line
                for line in lines
                if not line.strip().startswith("```")
            ]

            text = "\n".join(lines)

        start = text.find("{")
        end = text.rfind("}")

        if start == -1 or end == -1:
            raise ValueError(
                "AI evaluation response did not contain valid JSON."
            )

        return text[start:end + 1]
